## Imports

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import HTML
import joblib

if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")

## Functions

In [3]:
def extract_mediapipe_to_csv(data_path: str, save_path: str, model_path: str):

    video_extensions = [".mp4", ".mov", ".avi", ".mkv"]
    static = not any(data_path.lower().endswith(ext) for ext in video_extensions)

    base_options = python.BaseOptions(model_asset_path=model_path)

    JOINT_ORDER = [
        "head",
        "left_shoulder", "left_elbow",
        "right_shoulder", "right_elbow",
        "left_hand", "right_hand",
        "left_hip", "right_hip",
        "left_knee", "right_knee",
        "left_foot", "right_foot"
    ]

    LANDMARK_INDEX = {
        "head": 0,  
        "left_shoulder": 11,
        "right_shoulder": 12,
        "left_elbow": 13,
        "right_elbow": 14,
        "left_hand": 15,
        "right_hand": 16,
        "left_hip": 23,
        "right_hip": 24,
        "left_knee": 25,
        "right_knee": 26,
        "left_foot": 27,
        "right_foot": 28,
    }

    data = []

    if static:
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5
        )

        with vision.PoseLandmarker.create_from_options(options) as landmarker:
            image = cv2.imread(data_path)
            if image is None:
                raise ValueError(f"Could not read image: {data_path}")

            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
            results = landmarker.detect(mp_image)

            if results.pose_landmarks:
                row = {"FrameNo": 0}

                for joint in JOINT_ORDER:
                    idx = LANDMARK_INDEX[joint]
                    lm = results.pose_landmarks[0][idx]

                    row[f"{joint}_x"] = lm.x
                    row[f"{joint}_y"] = lm.y
                    row[f"{joint}_z"] = lm.z

                data.append(row)

    else:
        options = vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5
        )

        with vision.PoseLandmarker.create_from_options(options) as landmarker:
            cap = cv2.VideoCapture(data_path)

            if not cap.isOpened():
                raise ValueError(f"Could not open video: {data_path}")

            fps = cap.get(cv2.CAP_PROP_FPS)
            frame_idx = 0

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break

                image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

                timestamp_ms = int((frame_idx / fps) * 1000)
                results = landmarker.detect_for_video(mp_image, timestamp_ms)

                if results.pose_landmarks:
                    row = {"FrameNo": frame_idx}

                    for joint in JOINT_ORDER:
                        idx = LANDMARK_INDEX[joint]
                        lm = results.pose_landmarks[0][idx]

                        row[f"{joint}_x"] = lm.x
                        row[f"{joint}_y"] = lm.y
                        row[f"{joint}_z"] = lm.z

                    data.append(row)

                frame_idx += 1

            cap.release()

    # ---- SAVE CSV ----
    columns = ["FrameNo"]
    for joint in JOINT_ORDER:
        columns += [f"{joint}_x", f"{joint}_y", f"{joint}_z"]

    df = pd.DataFrame(data)
    df = df[columns]

    #os.makedirs(os.path.dirname(save_path), exist_ok=True)
    #df.to_csv(save_path, index=False)
    #print(f"Saved {len(df)} rows to {save_path}")

    return df



class Recurrent_classifier(nn.Module):
    def __init__(self, hidden_layers: list, layer_type="LSTM", dropout=0):
        super().__init__()

        input_size = 39
        rnn_class  = nn.LSTM if layer_type == "LSTM" else nn.GRU

        self.rnns  = nn.ModuleList()
        self.drops = nn.ModuleList()
        self.norms = nn.ModuleList()  # Add normalization layers

        sizes = [input_size] + hidden_layers
        for i in range(len(hidden_layers)):
            in_size = sizes[i] * 2 if i > 0 else sizes[i]  # *2 because bidirectional doubles output
            self.rnns.append(rnn_class(in_size, sizes[i + 1], batch_first=True, bidirectional=True))
            self.drops.append(nn.Dropout(dropout) if dropout > 0 else nn.Identity())
            
            # Add LayerNorm after each RNN (normalize over the hidden dimension)
            hidden_dim = sizes[i + 1] * 2  # *2 for bidirectional
            self.norms.append(nn.LayerNorm(hidden_dim))

        self.fc_out = nn.Linear(hidden_layers[-1] * 2, 1)  # *2 for bidirectional

    def forward(self, x):
        for rnn, drop, norm in zip(self.rnns, self.drops, self.norms):
            x, _ = rnn(x)
            x = norm(x)  # Apply layer normalization
            x = drop(x)
    
        return self.fc_out(x)  # (batch, seq_len, 1) — raw logits


def trim_csv_with_rnn_model(model, df, scaler, seq_length=30, stride=15, threshold=0.5):
    df = df.copy()
    df.columns = df.columns.str.strip()

    if "FrameNo" not in df.columns:
        df["FrameNo"] = np.arange(len(df))

    feature_cols = [c for c in df.columns
        if c.endswith("_x") or c.endswith("_y") or c.endswith("_z")]

    X_np = df[feature_cols].values.astype(np.float32)
    X_scaled = scaler.transform(X_np)

    probs_sum = np.zeros(len(df))
    counts = np.zeros(len(df))

    model.eval()
    with torch.no_grad():
        for i in range(0, len(df) - seq_length + 1, stride):
            seq = X_scaled[i:i + seq_length]

            X = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
            logits = model(X)

            probs = torch.sigmoid(logits).cpu().numpy().flatten()

            probs_sum[i:i + seq_length] += probs
            counts[i:i + seq_length] += 1

    frame_probs = probs_sum / np.maximum(counts, 1)
    preds = (frame_probs >= threshold).astype(int)

    print("Predictions:")
    print(preds)

    df["pred_running"] = preds

    movement_frames = df[df["pred_running"] == 1]

    if len(movement_frames) == 0:
        print("No movement detected")
        return None

    start_idx = movement_frames.index.min()
    stop_idx = movement_frames.index.max()

    trimmed_df = df.loc[start_idx:stop_idx].drop(columns=["pred_running"])

    print(f"Start index: {start_idx}, Stop index: {stop_idx}")

    return trimmed_df


In [4]:

def get_quality_label(file_name):
    """
    1 = good squat
    0 = bad squat
    """
    base = os.path.basename(file_name)

    if base.startswith("A") or base.startswith("G"):
        return 1
    elif base.startswith("W"):
        return 0
    else:
        raise ValueError(f"Unknown label for file: {file_name}")


def get_feature_columns(df):
    joints = [
        "head",
        "left_shoulder", "left_elbow",
        "right_shoulder", "right_elbow",
        "left_hand", "right_hand",
        "left_hip", "right_hip",
        "left_knee", "right_knee",
        "left_foot", "right_foot"
    ]

    feature_cols = []
    for joint in joints:
        feature_cols += [f"{joint}_x", f"{joint}_y", f"{joint}_z"]

    return feature_cols


def pad_sequence(df, max_frames):
    """
    Takes one trimmed video dataframe and pads it to max_frames.
    Output shape: (max_frames, 39)
    """

    feature_cols = get_feature_columns(df)

    X = df[feature_cols].values.astype(np.float32)

    n_frames = len(X)
    n_features = X.shape[1]

    padded_X = np.zeros((max_frames, n_features), dtype=np.float32)
    padded_X[:n_frames] = X

    return padded_X

In [ ]:
def build_padded_video_dataset(
    video_folder,
    output_folder,
    mediapipe_model_path,
    start_stop_model,
    start_stop_scaler,
    seq_length=30,
    stride=10,
    threshold=0.5
):
    """
    Full pipeline:

    1. Run MediaPipe on every video
    2. Trim each video using start/stop model
    3. Get target from filename
    4. Find longest trimmed sequence
    5. Pad all shorter sequences with zeros
    6. Save one CSV per video with readable feature names
    """

    os.makedirs(output_folder, exist_ok=True)

    video_extensions = [".mp4", ".mov", ".avi", ".mkv"]

    joints = [
        "head",
        "left_shoulder", "left_elbow",
        "right_shoulder", "right_elbow",
        "left_hand", "right_hand",
        "left_hip", "right_hip",
        "left_knee", "right_knee",
        "left_foot", "right_foot"
    ]

    processed_videos = []

    # =====================================================
    # FIRST PASS: extract + trim + store in memory
    # =====================================================

    for file_name in os.listdir(video_folder):

        if not any(file_name.lower().endswith(ext) for ext in video_extensions):
            continue

        print(f"\nProcessing: {file_name}")

        video_path = os.path.join(video_folder, file_name)

        # 1. Run MediaPipe
        df_mediapipe = extract_mediapipe_to_csv(
            data_path=video_path,
            save_path="",
            model_path=mediapipe_model_path
        )

        if df_mediapipe is None or len(df_mediapipe) == 0:
            print(f"Skipping {file_name}: no MediaPipe landmarks found")
            continue

        # 2. Trim with start/stop model
        trimmed_df = trim_csv_with_rnn_model(
            model=start_stop_model,
            df=df_mediapipe,
            scaler=start_stop_scaler,
            seq_length=seq_length,
            stride=stride,
            threshold=threshold
        )

        if trimmed_df is None or len(trimmed_df) == 0:
            print(f"Skipping {file_name}: no movement detected")
            continue

        # 3. Get label
        target = get_quality_label(file_name)

        processed_videos.append({
            "file_name": file_name,
            "trimmed_df": trimmed_df,
            "target": target,
            "length": len(trimmed_df)
        })

        print(f"Trimmed length: {len(trimmed_df)} frames")
        print(f"Target: {target}")

    if len(processed_videos) == 0:
        raise ValueError("No videos were successfully processed.")

    # =====================================================
    # FIND LONGEST SEQUENCE
    # =====================================================

    max_frames = max(video["length"] for video in processed_videos)

    print("\nLongest trimmed sequence:")
    print(f"{max_frames} frames")

    # =====================================================
    # CREATE READABLE COLUMN NAMES
    # =====================================================

    columns = []

    for frame_idx in range(max_frames):
        for joint in joints:
            columns += [
                f"frame{frame_idx}_{joint}_x",
                f"frame{frame_idx}_{joint}_y",
                f"frame{frame_idx}_{joint}_z"
            ]

    columns.append("target")

    # =====================================================
    # SECOND PASS: pad + save one CSV per video
    # =====================================================

    for video in processed_videos:

        file_name = video["file_name"]
        trimmed_df = video["trimmed_df"]
        target = video["target"]

        padded_X = pad_sequence(trimmed_df, max_frames=max_frames)

        row = padded_X.flatten().tolist()
        row.append(target)

        output_df = pd.DataFrame([row], columns=columns)

        base_name = os.path.splitext(file_name)[0]
        output_path = os.path.join(output_folder, f"{base_name}_padded.csv")

        output_df.to_csv(output_path, index=False)

        print(f"Saved: {output_path}")

    print("\nDone.")
    print(f"Saved {len(processed_videos)} padded video CSV files.")
    print(f"Each video padded to {max_frames} frames.")

## Paths

In [6]:
model_path = "../../MainProject/Assignment12/final_model_final.pt"
scaler_path = "../../MainProject/Assignment12/final_model_scaler.joblib"
video_folder_path = "../../../GoodvsBad"
output_folder = "../../MainProject/data/mediapipe_padded_videos"

## Load model

In [7]:
checkpoint = torch.load(model_path, map_location=device)

config = checkpoint["hyperparameters"]

model = Recurrent_classifier(
    hidden_layers=config["hidden_layers"],
    layer_type=config.get("layer_type", "LSTM"),
    dropout=config["dropout"]
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()



Recurrent_classifier(
  (rnns): ModuleList(
    (0): LSTM(39, 64, batch_first=True, bidirectional=True)
    (1): LSTM(128, 32, batch_first=True, bidirectional=True)
  )
  (drops): ModuleList(
    (0-1): 2 x Dropout(p=0.2, inplace=False)
  )
  (norms): ModuleList(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (fc_out): Linear(in_features=64, out_features=1, bias=True)
)

## Load scaler

In [8]:
scaler = joblib.load(scaler_path)


## Run trough videos

In [16]:
build_padded_video_dataset(
    video_folder=video_folder_path,
    output_folder=output_folder,
    mediapipe_model_path="../data/pose_landmarker.task",
    start_stop_model=model,
    start_stop_scaler=scaler,
    seq_length=config["seq_length"],
    stride=config["stride"],
    threshold=0.5
)


Processing: W28.avi


I0000 00:00:1778247030.321001 5695356 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1778247030.392975 5695362 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778247030.421095 5695362 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Predictions:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Start index: 25, Stop index: 85
Trimmed length: 61 frames
Target: 0

Processing: G43.avi


I0000 00:00:1778247033.489123 5695399 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1778247033.548960 5695402 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778247033.560086 5695403 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Predictions:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Start index: 60, Stop index: 109
Trimmed length: 50 frames
Target: 1

Processing: G57.avi


...

Predictions:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 1 1 1 1 1 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0]
Start index: 41, Stop index: 151
Trimmed length: 111 frames
Target: 0

Processing: W31.avi


I0000 00:00:1778247483.643369 5701767 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1778247483.703989 5701770 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778247483.712885 5701770 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Predictions:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Start index: 93, Stop index: 159
Trimmed length: 67 frames
Target: 0

Processing: G66.avi


I0000 00:00:1778247488.479031 5701810 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1778247488.540363 5701813 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778247488.549255 5701817 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Predictions:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]
Start index: 40, Stop index: 113
Trimmed length: 74 frames
Target: 1

Longest trimmed sequence:
173 frames
Saved: ../../MainProject/data/mediapipe_padded_videos/W28_padded.csv
Saved: ../../MainProject/data/mediapipe_padded_videos/G43_padded.csv
Saved: ../../MainProject/data/mediapipe_padded_videos/G57_padded.csv
Saved: ../../MainProject/data/mediapipe_padded_videos/G80_padded.csv
Saved: ../../MainProject/data/mediapipe_padded_videos/G81_padded.csv
Saved: ../../MainProject/data/mediapipe_padded_videos/G56_padded.csv
Saved: ../../MainProject/data/mediapipe_padded_videos/G42_padde

In [ ]:
df = pd.read_csv("../../MainProject/data/mediapipe_padded_videos/G01_padded.csv")

y = df["target"].values

X_flat = df.drop(columns=["target"]).values

max_frames = 173 
n_features = 39

X = X_flat.reshape(-1, max_frames, n_features)

print(X.shape)

print(X)
print(y)

(1, 173, 39)
[[[ 0.52720726  0.16865908 -0.32241583 ...  0.49026176  0.78741807
    0.16503   ]
  [ 0.52720386  0.16914926 -0.33219615 ...  0.48986819  0.78740412
    0.15604694]
  [ 0.52691746  0.1691364  -0.32683185 ...  0.4899978   0.792409
    0.15246266]
  ...
  [ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.          0.          0.         ...  0.          0.
    0.        ]]]
[1]
